# Seal5 Demo

This demo shows the creation of a C++ compiler with support for custom (RISCV) instructions to introduce an extension for enabling the ChaCha20 steam cipher on edge devices. It walks through the Seal5 flow, which consists of:
1. Setting up the Seal5 environment
2. Priming the extensible compiler (based on Clang/LLVM)
3. Initial build of the (baseline) compiler
4. Compliling and executing the (baseline) ChaCha20 test application
6. Creation and application of patches based on an instruction set description (in CoreDSL)
7. Building and verification of the extended compiler
8. Building and executing the optimized ChaCha20 test application
9. Comparison of the results

## ChaCha20 - a motivating use case
The [ChaCha20 stream cipher](https://en.wikipedia.org/wiki/Salsa20#ChaCha_variant) is designed for high throughput on simple processors, by repeatedly applying only additions, exclusive-ors, and bit rotations (ADD+XOR+ROL) across a 4 * 4 array of 32-bit words to generate a high-quality stream of pseudo-random bytes for enciphering the data. 

A "quarter round" is a set of four successive sets of ADD+XOR+ROL:

```
void quarter_round(uint32_t* v, int a, int b, int c, int d) {
    v[a] += v[b]; v[d] ^= v[a]; v[d] = rol(v[d], 16);
    v[c] += v[d]; v[b] ^= v[c]; v[b] = rol(v[b], 12);
    v[a] += v[b]; v[d] ^= v[a]; v[d] = rol(v[d],  8);
    v[c] += v[d]; v[b] ^= v[c]; v[b] = rol(v[b],  7);
}
```

 and ChaCha20 performs 20 complete rounds across 4 colums and separately 4 diagonals of the array, resulting in 80 invocations of quarter_round(). Although attempting to optimize without profiling is risky, here we can guess that the overall performance will be strongly coupled to the quarter_round implementation. 
 
For validating a ChaCha20 implementation the published specification includes the 16-word array generated from an initial NULL block. The test code used in this demo confirms that the correct array state is produced as well as reporting the number of (simulated) clock cycles consumed, as will be seen below.

Building this code for a naive RV32I core (standard RISC-V minimally-featured core: 32 x 32-bit registers, only basic airthmetic and logic operations) shows that each set of ADD+XOR+ROL operations becomes these five RV32I instructions: 
```
add     x9, x9, x8
xor     x5, x5, x9
srli    x21, x5, 0x10
slli    x5, x5, 0x10
or      x5, x5, x21
```
with the ROL supported by the OR of two separate shifts. 



This code seems tight; there is no obvious way to improve upon it with the instructions available, so we can consider what additional instructions could be useful. A quick literature search shows that high-performance adder circuits are an enduring research field, generating complex designs. Creating a bespoke adder for ChaCha20 may be hard to justify. However bitwise-XOR is almost trivial, and a fixed-length bit rotation is equally simple, so we can hope that a custom instruction combining the XOR+ROL will be affordable. Furthermore, since the same value is updated by both the combined instruction should be simple for the compiler to select.

In the rest of this demo we implement this idea by building the toolchain with Seal5 to confirm  the naive RV32I behavior, add custom instructions to combine XOR+ROL, and then demonstrate that the newly-extended toolchain then produces much improved code.

## Preparation

Seal5 is written in Python. To use it, you must first import various Seal5 features (e.g. classes) as done below.

In [ ]:
import os
import jt
from IPython.display import HTML, display
from pathlib import Path
from seal5.flow import Seal5Flow
from seal5.logging import set_log_level
from seal5.types import PatchStage

To receive some meaningful output from the following steps, you should configure the log level with the following command. For this demo, we will disable most of the output. It will still be present in the logfile though.

In [ ]:
set_log_level(console_level="ERROR", file_level="DEBUG")

Now it's time to create our Seal5Flow object that we will use during the rest of the process. The first argument specifies the name of the folder that will be used to download all the necessary files, patch and build the extended compiler.

In [ ]:
seal5_flow = Seal5Flow('seal5-demo', name='demo')

The following two commands are optional and can be used to clean and reset various aspects of the flow (e.g. delete the build folder). Run them e.g. if you did some experiments and want to start over clean.

In [ ]:
seal5_flow.reset(settings=True, interactive=False)
seal5_flow.clean(temp=True, patches=True, models=True, inputs=True, interactive=False)

## Downloading LLVM

The following command will download an appropriate version of LLVM into the previously configured folder, that will be used as a foundation for our custom compiler. For this demo, we clone from a local repository to not depend on network availability.

In [ ]:
seal5_flow.initialize(
    clone=True,
    clone_url=str((Path() / "llvm-project.git").resolve()),
    clone_ref="llvmorg-20.1.0",
    clone_depth=1,
    progress=True,
    force=True,
    verbose=True,
)

## Loading the Definitions

Seal5 can load multiple definitions (e.g. CoreDSL inputs) into the same object to create a combined model and (based on that) compiler extension. In addtion to core DSL inputs, you may also specify test cases and overrides for various CoreDSL definitions and other setting using a custom (YAML) format. For now let's tell Seal5 about our custom instructions; we'll have a look at those later before they will be applied:

In [ ]:
EXAMPLES_DIR = Path() / "seal5" / "examples"
seal5_flow.load([
    EXAMPLES_DIR / "chacha20" / "cdsl" / "chacha20_llvm.core_desc"
    ], verbose=False, overwrite=True
)

Test cases will be executed using [LLVM Lit](https://llvm.org/docs/CommandGuide/lit.html). As such, various different inputs are supported as test cases, e.g. assembly, LLVM IR and C/C++ files. The the documentation of LLVM Lit for further information.
Our example comes with various test cases, let's add them all to the flow:

In [ ]:
TESTS_DIR = EXAMPLES_DIR / "chacha20" / "tests"
seal5_flow.load([
    TESTS_DIR / "*.c",
    TESTS_DIR / "*.s",
    ], verbose=False, overwrite=True
)

Run the following command to see the content of the test case in C syntax as an example (change the file name to view the others):

In [ ]:
!cat "$TESTS_DIR/chacha20_xorrol16.test-builtin.c"

Let's also load various setting from the supplied YAML files to configure our build correctly using the command below:

In [ ]:
seal5_flow.load([
    EXAMPLES_DIR / "common" / "cfg" / "llvm.yml",
    EXAMPLES_DIR / "common" / "cfg" / "filter.yml",
    EXAMPLES_DIR / "common" / "cfg" / "patches.yml",
    EXAMPLES_DIR / "common" / "cfg" / "riscv.yml",
    EXAMPLES_DIR / "common" / "cfg" / "tests.yml",
    EXAMPLES_DIR / "common" / "cfg" / "passes.yml",
    EXAMPLES_DIR / "common" / "cfg" / "git.yml",
    EXAMPLES_DIR / "chacha20" / "cfg" / "intrinsics.yml",
    ], verbose=False, overwrite=False
)

Again, let's inspect one of those files to see what they are about. For our demo, *filter.yml* is used to prevent Seal5 from generating code for the basic RISCV instructions, since those are alreay part of LLVM without any modification: 

In [ ]:
!cat "$EXAMPLES_DIR/common/cfg/filter.yml"

## Initial building and priming of LLVM

The following steps configure the build process of LLVM (e.g. build in release mode with debug assertions), apply initial patches (e.g. provide insertion markers ontop of the upstream LLVM release that will later allow applying the generated patches) and build the initial compiler:

In [ ]:
seal5_flow.settings.llvm.default_config = 'release_assertions'
seal5_flow.settings.tools.pattern_gen.clone_url=str((Path() / "CoreDSL2LLVM.git").resolve())
seal5_flow.setup(force=True, progress=True, verbose=False)
seal5_flow.patch(verbose=False, stages=[PatchStage.PHASE_0])
seal5_flow.build(verbose=True, config='release_assertions', enable_ccache=True)

## Compiling the baseline ChaCha20 demo application

With the now available (baseline) LLVM compiler, we can build our baseline application to later benchmark the optimized application against.

First, we ask Seal5 where our just compiled compiler is in our filesystem so that we can make use of it. We also need a GCC toolchain as a foundation (e.g. glibc), that we have lying around (creating that is not part of this demo).

In [ ]:
RISCV_GCC = (Path() / "riscv_gcc").resolve()
XCLANG = seal5_flow.settings.get_llvm_build_dir('release_assertions') / "bin"

For this demo, we have a simple shell script that wraps the process of creating build directories, setting up the CMake configuration and compiling the demo application. Let's build two versions of it optimized for speed (O3) and code size (Os) respectively. The file "rv32gc-llvm-toolchain.cmake" is a so called CMake toolchain file that contains information on how to use our baseline compiler.

In [ ]:
!scripts/build_chacha.sh "$RISCV_GCC" "$XCLANG" "rv32gc-llvm-toolchain.cmake" "baseline-o3" "O3"
!scripts/build_chacha.sh "$RISCV_GCC" "$XCLANG" "rv32gc-llvm-toolchain.cmake" "baseline-os" "Os"

Before doing anything with our applications, let's look at the code size. We have a simple shell script that inspects our binaries using GCC's binutils and strips all unnecessary symbols for a fair comparison for that purpose:

In [ ]:
!scripts/chacha_code_size.sh "$RISCV_GCC" "baseline"

## Running the (baseline) application

We actually run our application using an instruction set simulator (ISS) let's launch it for the speed optimized version first and have a look at it using vscode. One interesting position to set the breakpoint is the function `quarter_round` inside the source file `chacha20/main.c`. The ISS will also save runtime information (e.g. cycle count) to a JSON file.

In [ ]:
!scripts/run_chacha.sh "baseline-o3" -pgdbserver --plugin.gdbserver.port=2222

Let's also run the size optimized version to determine the cycle count for that as well.

In [ ]:
!scripts/run_chacha.sh "baseline-os"

We have a small script to print the relevant information from those JSON outputs:

In [ ]:
!scripts/chacha_cycle_count.sh "baseline"

## Optimizing the application

Now that we have an idea how our application spents its time, let's introduce our custom instructions to speed things up.
Normally, you would profile the baseline application to come up with these instructions. For this demo, we already have some that were already provided to Seal5 above. Let's have a look at one of those: the *CHACHA20QR1* instruction. This is an excerpt from its definition in CoreDSL:

First, we import the basic RISCV core definition that our extension will by applied onto:

`01` `import "../rv_base/RV32I.core_desc"`

Now we define our instruction set *XChaCha* based on it:

`03` `InstructionSet XCHACHA extends RV32I {`  
`04` `  instructions {`

The four sets of ADD+XOR+ROL operations in a quarter_round all have different rotation sizes, so we will need four custom instructions. Here's the first:

`05` `    CV_CHACHA20QR1 {`

An instruction is described by its encoding and assembly syntax, in this cases, three register indices (*rd*, *rs1* and *rs2*) are encoded in reverse order:

`06` `      encoding: 7'b0000000 :: rs2[4:0] :: rs1[4:0] :: 3'b000 :: rd[4:0] :: 7'b0001011;`  
`07` `      "xexample.chacha20qr1, {name(rd)}, {name(rs1)}, {name(rs2)}";`

... as well as its behavior, which is used to drive ISS and hardware generation, as well as automatic selection by the compiler (if possible). So here the instruction implements the operations "d ^= a; d <<<= 16;", potentially replacing four RV32I instructions with just this one::

`08` `      behavior: {`  
`09` `         if (rd != 0) {`   
`10` `            unsigned<32> xor = X[rs1] ^ X[rs2];`   
`11` `            X[rd] = (xor << 16) | (xor >> 16);`   
          
The rest of the file is just closing all the brackets:

`12` `        }`  
`13` `      }`  
`14` `    }`  
`15` `  }`  
`16` `}`

### Generating the LLVM patches

The following commands will transform all the loaded inputs into a consistent model and then generate a patch for LLVM that adds our new instructions, including *CHACHA20QR1*:

In [ ]:
seal5_flow.transform(verbose=False)
seal5_flow.generate(verbose=False, skip=["pattern_gen"])

### Patching and building the compiler

The following commands will apply our patches to add our extension to the LLVM base compiler and re-build only the neccessary files.

In [ ]:
seal5_flow.patch(verbose=False, stages=[PatchStage.PHASE_1, PatchStage.PHASE_2])
seal5_flow.build(verbose=True, config='release_assertions', enable_ccache=True)

### Automatic instruction selection

The following commands will add pattern gen support for our custom instructions, such that they can bes selected by the compiler and perform a final rebuild of all the files that were modified during that process:

In [ ]:
seal5_flow.build(verbose=True, config='release_assertions', target="pattern-gen", enable_ccache=True)
seal5_flow.build(verbose=True, config='release_assertions', target="llc", enable_ccache=True)
# Generate remaining patches
seal5_flow.generate(verbose=False, only=["pattern_gen"])
# Apply patches
seal5_flow.patch(verbose=False, stages=list(range(PatchStage.PHASE_3, PatchStage.PHASE_5 + 1)))
seal5_flow.build(verbose=True, config='release_assertions', enable_ccache=True)

## Verification

A simple first test is to manually launch our extended compiler and check whether our new extension is actually available:

In [ ]:
%%bash -s "$XCLANG"
PATH=$1:$PATH
clang++ --print-supported-extensions 2> /dev/null | grep xchacha

The following commands will execute all the test cases we have previously added to the flow using LLVM Lit:

In [ ]:
seal5_flow.test(verbose=False, ignore_error=False)

We have a small script to render our test results, let's executed it and have a look:

In [ ]:
!scripts/gen_reports.sh "seal5-demo" &> /dev/null
with open('Grouped_stat_prop_result_test.html', 'r', encoding='utf-8') as f:
    html_content = f.read()
display(HTML(html_content))

## Building the (optimized) demo application

Building the optimized application generally works the same way. We use a different toolchain file that enables our extension for our compiler, other than that, we can just execute the same commands:

In [ ]:
!scripts/build_chacha.sh "$RISCV_GCC" "$XCLANG" "rv32gc-xchacha-llvm-toolchain.cmake" "optimized-o3" "O3"
!scripts/build_chacha.sh "$RISCV_GCC" "$XCLANG" "rv32gc-xchacha-llvm-toolchain.cmake" "optimized-os" "Os"

With that, we can now compare the size with the baseline application:

In [ ]:
!scripts/chacha_code_size.sh "$RISCV_GCC" "baseline"
!scripts/chacha_code_size.sh "$RISCV_GCC" "optimized"

## Running the (optimized) application

Again, we execute our ISS and have a look at the assembly with vscode:

In [ ]:
!scripts/run_chacha.sh "optimized-o3" -pgdbserver --plugin.gdbserver.port=2222

And let's also run the size optimized version, without attaching to the process:

In [ ]:
!scripts/run_chacha.sh "optimized-os"

Using our script, we can now compare the cycle count against our baseline application:

In [ ]:
!scripts/chacha_cycle_count.sh "baseline"
!scripts/chacha_cycle_count.sh "optimized"